# Ориентация кропа с текстом (0° / 180°): ансамбль из трёх моделей

**Задача.** Для каждого из 20 000 кропов предсказать `p_180`. Метрика: 1 − Brier, поэтому важны калиброванные вероятности.

## Подход
1. **Две готовые модели PaddleOCR** (без обучения): `PP-LCNet_x0_25_textline_ori` (1.1 МБ) и `PP-LCNet_x1_0_textline_ori` (6.6 МБ).
2. **Своя компактная модель** `mobilenetv3_small_100` (timm, ImageNet-инициализация, 1.5M параметров, 6.1 МБ в ONNX), один логит, вход 48×256.
   Обучена на кропах со **случайным поворотом на 180°** (метка получается бесплатно): TextOCR (реальные сцены),
   кириллица `DonkeySmall/OCR-Cyrillic-Printed-10` и PIL-синтетика (133 шрифта с кириллицей: слова, цены, числа).
   Аугментации: масштаб по ширине, обрезка краёв, поворот/перспектива, яркость/контраст, ч/б, инверсия, blur, шум, JPEG, даунскейл.
3. **TTA в пространстве логитов**: `z = 0.5·(z(x) − z(rot180(x)))`. Усреднение логитов вместо вероятностей заметно лучше по Brier
   (для Paddle x0_25 на ICDAR: 0.064 против 0.095).
4. **Стэкинг**: логистическая регрессия по трём TTA-логитам (3 веса + смещение), обучена **только на отложенной реальной валидации**.
   Это одновременно калибровка вероятностей. Тест не используется ни для обучения, ни для калибровки, псевдоразметки нет.

## Препроцессинг
Ресайз по высоте до 48 px с сохранением пропорций, ширина не более 256 (длинные кропы сжимаются), справа серый паддинг.
Поворот на 180° делается **до** паддинга, поэтому обучение и инференс согласованы. Вертикальных кропов (h > w) в тесте нет
(проверено по выборке), в обучении отбрасываются кропы с aspect ratio < 1.5.

## Валидация
Split по **источнику**, а не случайный: ICDAR2015 (реальные сцены, не участвует в обучении) и hold-out кириллицы (3000 кропов).
Метки поворота на валидации фиксируются по seed. Варианты ансамбля сравнивались по 5-fold Brier на этой валидации.

| Вариант | Brier на валидации | 1 − Brier на тесте (платформа) |
|---|---|---|
| Paddle x0_25, TTA по вероятностям (v1) | ~0.093 | 0.9265 |
| Своя модель, TTA + температура (v2) | 0.082 | 0.8754 |
| Paddle x0_25 + своя (v3b) | 0.043 | 0.9420 |
| **Paddle x0_25 + Paddle x1_0 + своя (v4, итог)** | **0.028** | **0.9625** |

Пробовала и отбросила: docTR `crop_orientation` (на кириллице точность 54%, в ансамбле выигрыша нет: 0.0281 → 0.0280).

## Скорость и размер своей модели (ONNX 6.08 МБ, 1.52M параметров)
| | batch 1 | batch 256 |
|---|---|---|
| CPU (onnxruntime) | 0.50 мс/кроп | 0.14 мс/кроп |
| GPU RTX 5060 (torch fp16) | 5.8 мс/кроп | 0.042 мс/кроп |

Обе Paddle-модели маленькие (1.1 и 6.6 МБ); инференс идёт с TTA (два прохода на модель).

## Использованные открытые компоненты
PyTorch, timm (MobileNetV3), PaddleOCR / PaddlePaddle / PaddleX (модели `PP-LCNet_*_textline_ori`), OpenCV, Pillow, scikit-learn, pandas, onnx, onnxruntime.
Датасеты: [MiXaiLL76/TextOCR_OCR](https://huggingface.co/datasets/MiXaiLL76/TextOCR_OCR) (TextOCR),
[MiXaiLL76/ICDAR2015_OCR](https://huggingface.co/datasets/MiXaiLL76/ICDAR2015_OCR),
[DonkeySmall/OCR-Cyrillic-Printed-10](https://huggingface.co/datasets/DonkeySmall/OCR-Cyrillic-Printed-10), синтетика PIL на шрифтах Windows.

## Воспроизводимость и ограничения
- Seed фиксирован (`config.py`: python, numpy, torch, воркеры DataLoader). Ансамбль детерминирован от запуска к запуску.
- Paddle и torch конфликтуют по CUDA-DLL в одном процессе, поэтому Paddle-признаки считаются в отдельном окружении `.venv_paddle`.
- Батчевый инференс PaddleOCR на кропах разного размера даёт неверные классы, используется `batch_size=1`.
- `submission.csv` воспроизводится кодом; от отправленного на платформу `submission_v4.csv` отличается не более чем на 0.03
  у 57 из 20 000 кропов (недетерминированность GPU-ядер в bf16 между запусками), на метрику это практически не влияет.
- Обучение `train.py` даёт близкие, но не побитово идентичные веса (аугментации в воркерах); для точного воспроизведения
  результата лежат веса `cache/model.pt`.
- Своя модель недообучена (Brier на валидации ещё падал); дообучение на большем числе реальных кропов — следующий шаг.
- В проде ансамбль из трёх моделей с TTA дороже одной; при жёстких требованиях к скорости можно оставить пару
  `Paddle x1_0 + своя` (5-fold Brier 0.0289 против 0.0281).


In [ ]:
# Общие настройки: seed и проверка GPU. Все параметры (пути, размеры, seed) лежат в config.py.
import os, subprocess, sys
import torch, config as C
C.seed_everything()
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

def run(*args, py=sys.executable, **env):
    """Запуск скрипта проекта подпроцессом (нужен, чтобы использовать Paddle-окружение)."""
    subprocess.run([py, *args], check=True, cwd=C.ROOT, env={**os.environ, **env})

## 1. Данные
Скачивание реальных кропов (TextOCR, ICDAR, кириллица) с Hugging Face, генерация PIL-синтетики, кэш в `cache/*.pkl`, валидационные наборы `cache/val_*.pkl`. Повторный запуск берёт кэш.

In [ ]:
run('data_prep.py')

## 2. Обучение своей модели
`mobilenetv3_small_100`, BCEWithLogits, AdamW + OneCycle, bf16-AMP, 3 эпохи (~16 мин на RTX 5060). Каждый пример получает случайный поворот на 180° и соответствующую метку. Лучший чекпойнт по Brier на ICDAR сохраняется в `cache/model.pt`. Если веса уже есть, шаг пропускается.

In [ ]:
if not os.path.exists(f'{C.CACHE}/model.pt'):
    run('train.py')
else:
    print('cache/model.pt уже есть, обучение пропущено')

## 3. Признаки готовых моделей PaddleOCR (отдельное окружение)
Paddle и torch конфликтуют по CUDA-DLL, поэтому создаётся `.venv_paddle` (см. `requirements_paddle.txt`). Считаются вероятности p(180°) для кропа и его поворота на 180° на валидации и на тесте; `batch_size=1`, так как батчи разных размеров ломают Paddle.

In [ ]:
VENV = os.path.join(C.ROOT, '.venv_paddle', 'Scripts' if os.name == 'nt' else 'bin', 'python')
if not os.path.exists(VENV):
    subprocess.run([sys.executable, '-m', 'venv', os.path.join(C.ROOT, '.venv_paddle')], check=True)
    subprocess.run([VENV, '-m', 'pip', 'install', 'paddlepaddle-gpu==3.2.0',
                    '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu129/'], check=True)
    subprocess.run([VENV, '-m', 'pip', 'install', 'paddleocr==3.7.0', 'paddlex==3.7.2',
                    'opencv-python-headless', 'pandas', 'numpy'], check=True)
for name, out in [('PP-LCNet_x0_25_textline_ori', 'paddle_raw.npz'),
                  ('PP-LCNet_x1_0_textline_ori', 'paddle_PP-LCNet_x1_0_textline_ori.npz')]:
    if not os.path.exists(f'{C.CACHE}/{out}'):
        run('paddle_logits.py', name, py=VENV, PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK='True')
    else:
        print(out, 'уже есть')

## 4. Ансамбль, калибровка и `submission.csv`
Логит-TTA каждой модели -> логистическая регрессия, обученная на реальной валидации (ICDAR + hold-out кириллицы). Печатается качество одиночных моделей и вариантов стэка (5-fold). Тест только применяется, в подборе не участвует.

In [ ]:
run('ensemble.py', 'submission.csv')

## 5. Проверка формата и sanity check по тесту

In [ ]:
import pandas as pd, numpy as np
s, ref = pd.read_csv(f'{C.ROOT}/submission.csv'), pd.read_csv(C.SAMPLE_SUB)
assert list(s.columns) == ['image_id', 'p_180'] and len(s) == 20000 and (s.image_id == ref.image_id).all()
assert s.p_180.between(0, 1).all() and not s.p_180.isna().any()
print('формат ок; доля p>0.5 =', round((s.p_180 > .5).mean(), 3), '| гистограмма:', np.histogram(s.p_180, 10, (0, 1))[0])

## 6. ONNX, размер и латентность своей модели

In [ ]:
run('bench.py')